# Consistency in L features

In [ ]:
import os
import re
import glob
from typing import List, Optional, Tuple, Dict, Literal

import numpy as np
import pandas as pd
from scipy import stats

# -------------------------
# Generic directory loader (summary_sc_<id>.csv OR turns_<id>.csv)
# -------------------------
def load_participant_matrix_from_dir(
    data_dir: str,
    *,
    pattern: str,
    id_regex: str,
    features: Optional[List[str]] = None,
    # for one-row files use "first"; for turns_<id>.csv use "median" (recommended)
    row_agg: Literal["first", "mean", "median"] = "first",
    # participant-only turns filter (best-effort heuristic)
    participant_only: bool = True,
    speaker_col_candidates: Tuple[str, ...] = (
        "speaker", "Speaker", "role", "Role", "speaker_role", "SpeakerRole", "who", "Who"
    ),
    # values that indicate interviewer/Ellie to exclude
    exclude_speakers_substrings: Tuple[str, ...] = ("ellie", "interviewer", "assistant"),
) -> pd.DataFrame:
    """
    Reads many CSVs like summary_sc_<id>.csv or turns_<id>.csv into one DataFrame:
      index = Participant (id from filename)
      columns = aggregated features

    For turns_<id>.csv (multiple rows), set row_agg="median" (default recommendation).
    """
    if not os.path.isdir(data_dir):
        raise FileNotFoundError(f"Not a directory: {data_dir}")

    files = sorted(glob.glob(os.path.join(data_dir, pattern)))
    if not files:
        raise FileNotFoundError(f"No files found: {os.path.join(data_dir, pattern)}")

    rx = re.compile(id_regex)
    rows = []
    for fp in files:
        m = rx.search(os.path.basename(fp))
        if not m:
            continue
        pid = int(m.group(1))

        df = pd.read_csv(fp)
        if df.empty:
            continue

        # optional: keep participant-only turns by removing Ellie/interviewer rows
        if participant_only:
            spk_col = next((c for c in speaker_col_candidates if c in df.columns), None)
            if spk_col is not None:
                spk = df[spk_col].astype(str).str.lower()
                mask = np.ones(len(df), dtype=bool)
                for bad in exclude_speakers_substrings:
                    mask &= ~spk.str.contains(bad, na=False)
                df = df.loc[mask]
                if df.empty:
                    continue

        # keep only requested features (+ any needed for filtering already done)
        if features is not None:
            keep = [c for c in features if c in df.columns]
            df = df[keep].copy()

        # convert columns to numeric where possible
        for c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

        if df.empty:
            continue

        if row_agg == "first":
            rec = df.iloc[0].to_dict()
        elif row_agg == "mean":
            rec = df.mean(axis=0, numeric_only=True).to_dict()
        elif row_agg == "median":
            rec = df.median(axis=0, numeric_only=True).to_dict()
        else:
            raise ValueError("row_agg must be one of: 'first','mean','median'")

        rec["Participant"] = pid
        rows.append(rec)

    if not rows:
        raise ValueError(f"Parsed 0 valid files in: {data_dir}")

    out = pd.DataFrame(rows).set_index("Participant").sort_index()
    return out


# -------------------------
# Agreement metrics (EN vs UA)
# -------------------------
def _paired_clean(a: np.ndarray, b: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    m = np.isfinite(a) & np.isfinite(b)
    return a[m], b[m]


def _bootstrap_ci(values_fn, a: np.ndarray, b: np.ndarray, n_boot: int = 2000, seed: int = 1706) -> Tuple[float, float, float]:
    rng = np.random.default_rng(seed)
    a, b = _paired_clean(a, b)
    n = len(a)
    if n < 3:
        return np.nan, np.nan, np.nan

    stats_list = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        s = values_fn(a[idx], b[idx])
        if np.isfinite(s):
            stats_list.append(s)

    if not stats_list:
        return np.nan, np.nan, np.nan

    lo, hi = np.percentile(stats_list, [2.5, 97.5])
    return float(np.mean(stats_list)), float(lo), float(hi)


def _pearson(a: np.ndarray, b: np.ndarray) -> float:
    a, b = _paired_clean(a, b)
    if len(a) < 3 or np.allclose(a, a[0]) or np.allclose(b, b[0]):
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def _spearman(a: np.ndarray, b: np.ndarray) -> float:
    a, b = _paired_clean(a, b)
    if len(a) < 3:
        return np.nan
    r, _ = stats.spearmanr(a, b, nan_policy="omit")
    return float(r)


def icc2_1(a: np.ndarray, b: np.ndarray) -> float:
    """
    ICC(2,1): two-way random effects, absolute agreement, single measurement.
    EN and UA are the two "raters"; subjects are participants.
    """
    a, b = _paired_clean(a, b)
    n = len(a)
    k = 2
    if n < 3:
        return np.nan

    Y = np.vstack([a, b]).T  # (n, 2)
    mean_subject = Y.mean(axis=1, keepdims=True)
    mean_rater = Y.mean(axis=0, keepdims=True)
    grand_mean = Y.mean()

    SSR = k * np.sum((mean_subject - grand_mean) ** 2)
    SSC = n * np.sum((mean_rater - grand_mean) ** 2)
    SSE = np.sum((Y - mean_subject - mean_rater + grand_mean) ** 2)

    MSR = SSR / (n - 1)
    MSC = SSC / (k - 1)
    MSE = SSE / ((n - 1) * (k - 1))

    denom = MSR + (k - 1) * MSE + (k * (MSC - MSE) / n)
    if denom <= 0:
        return np.nan
    return float((MSR - MSE) / denom)


def _mean_shift(a: np.ndarray, b: np.ndarray) -> float:
    a, b = _paired_clean(a, b)
    if len(a) == 0:
        return np.nan
    return float(np.mean(b - a))  # UA - EN


def _median_min_max(x: np.ndarray) -> Tuple[float, float, float]:
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan, np.nan, np.nan
    return float(np.median(x)), float(np.min(x)), float(np.max(x))


# -------------------------
# Main API (works for summary_sc_*.csv AND turns_*.csv)
# -------------------------
def check_statistic(
    *,
    features: List[str],
    dataset_eng_dir: str,
    dataset_ukr_dir: str,
    file_kind: Literal["summary", "turns"] = "summary",
    n_boot: int = 2000,
    seed: int = 1706,
    strict: bool = False,
    # turns only: how to aggregate multiple turn rows -> one value per participant per feature
    turns_agg: Literal["mean", "median"] = "median",
    # turns only: try to exclude interviewer/Ellie rows
    participant_only_turns: bool = True,
) -> pd.DataFrame:
    """
    Computes EN vs UA agreement per feature:
      Pearson r, Spearman rho, ICC(2,1), mean shift Δμ = mean(UA-EN), with bootstrap 95% CI.

    If file_kind="turns", we first aggregate each turns_<id>.csv into a single row per participant
    using turns_agg (default median). This matches your “turn-level -> interview-level” aggregation logic.
    """
    if file_kind == "summary":
        pattern = "summary_sc_*.csv"
        id_regex = r"summary_sc_(\d+)\.csv$"
        row_agg = "first"
        participant_only = False  # summary is already participant-only
    elif file_kind == "turns":
        pattern = "turns_*.csv"
        id_regex = r"turns_(\d+)\.csv$"
        row_agg = turns_agg
        participant_only = participant_only_turns
    else:
        raise ValueError("file_kind must be 'summary' or 'turns'")

    df_en = load_participant_matrix_from_dir(
        dataset_eng_dir,
        pattern=pattern,
        id_regex=id_regex,
        features=features,
        row_agg=row_agg,
        participant_only=participant_only,
    )
    df_ua = load_participant_matrix_from_dir(
        dataset_ukr_dir,
        pattern=pattern,
        id_regex=id_regex,
        features=features,
        row_agg=row_agg,
        participant_only=participant_only,
    )

    merged = df_en.add_suffix("_EN").join(df_ua.add_suffix("_UA"), how="inner")
    if merged.empty:
        raise ValueError("No overlapping participants between EN and UA dirs.")

    rows: List[Dict[str, object]] = []

    for feat in features:
        c_en = f"{feat}_EN"
        c_ua = f"{feat}_UA"

        if c_en not in merged.columns or c_ua not in merged.columns:
            if strict:
                raise KeyError(f"Missing feature in merged data: {feat}")
            rows.append({
                "feature": feat,
                "n": 0,
                "EN_median[min,max]": "",
                "UA_median[min,max]": "",
                "r [95% CI]": "",
                "rho [95% CI]": "",
                "ICC(2,1) [95% CI]": "",
                "mean shift Δμ (UA-EN) [95% CI]": "",
                "note": "missing in EN/UA files",
            })
            continue

        a = merged[c_en].to_numpy(dtype=float)  # EN
        b = merged[c_ua].to_numpy(dtype=float)  # UA
        a, b = _paired_clean(a, b)
        n = len(a)

        en_med, en_min, en_max = _median_min_max(a)
        ua_med, ua_min, ua_max = _median_min_max(b)

        r_m, r_lo, r_hi = _bootstrap_ci(_pearson, a, b, n_boot=n_boot, seed=seed)
        s_m, s_lo, s_hi = _bootstrap_ci(_spearman, a, b, n_boot=n_boot, seed=seed)

        icc_mean, icc_lo, icc_hi = _bootstrap_ci(lambda x, y: icc2_1(x, y), a, b, n_boot=n_boot, seed=seed)
        sh_m, sh_lo, sh_hi = _bootstrap_ci(_mean_shift, a, b, n_boot=n_boot, seed=seed)

        rows.append({
            "feature": feat,
            "n": int(n),
            "EN_median[min,max]": f"{en_med:.3f} [{en_min:.3f},{en_max:.3f}]",
            "UA_median[min,max]": f"{ua_med:.3f} [{ua_min:.3f},{ua_max:.3f}]",
            "r [95% CI]": f"{r_m:.3f} [{r_lo:.3f},{r_hi:.3f}]",
            "rho [95% CI]": f"{s_m:.3f} [{s_lo:.3f},{s_hi:.3f}]",
            "ICC(2,1) [95% CI]": f"{icc_mean:.3f} [{icc_lo:.3f},{icc_hi:.3f}]",
            "mean shift Δμ (UA-EN) [95% CI]": f"{sh_m:.3f} [{sh_lo:.3f},{sh_hi:.3f}]",
            "note": "",
        })

    out = pd.DataFrame(rows).sort_values(["n", "feature"], ascending=[False, True]).reset_index(drop=True)
    return out


In [11]:
# ---- B0: language-independent ----
B0_SUMMARY = [
    "file_length",
    "speech_length_minutes",
    "speech_length_words",
    "words_per_min",
    "speech_percentage",
    "mean_pre_word_pause",
    "mean_pause_variability",
    "word_repeat_percentage",
    "phrase_repeat_percentage",
    # optional session/turn aggregates (still language-independent)
    "num_turns",
    "num_one_word_turns",
    "mean_turn_length_minutes",
    "mean_turn_length_words",
    "mean_pre_turn_pause",
    "speaker_percentage",
    "num_interrupts",
]

stats_df = check_statistic(
    features=B0_SUMMARY,
    dataset_eng_dir="tmp/result_phonoma_gemma_eng_26-03-2026",#"/Users/pelmeshek1706/Desktop/projects/airest_notebooks/result_oppenwillis_eng_norm_gemma",
    dataset_ukr_dir="tmp/result_phonoma_gemma_ukr_26-03-2026",#"/Users/pelmeshek1706/Desktop/projects/airest_notebooks/result_oppenwillis_ukr_norm_gemma",
    file_kind="summary",
    n_boot=2000,
    seed=1706,
)
stats_df.drop(columns=["note", 'n'], inplace=True)
stats_df

/Users/pelmeshek1706/Desktop/projects/final_airest_voice/airest/.venv/lib/python3.10/site-packages/scipy/stats/_stats_py.py:5445: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(warn_msg))


,feature,"EN_median[min,max]","UA_median[min,max]",r [95% CI],rho [95% CI],"ICC(2,1) [95% CI]",mean shift Δμ (UA-EN) [95% CI]
0,file_length,"14.914 [6.785,32.395]","14.914 [6.785,32.395]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","0.000 [0.000,0.000]"
1,mean_pause_variability,"0.187 [0.023,5.651]","0.000 [0.000,0.000]","nan [nan,nan]","nan [nan,nan]","0.000 [-0.000,0.000]","-0.353 [-0.424,-0.292]"
2,mean_pre_turn_pause,"1.407 [0.492,4.476]","1.407 [0.492,4.476]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","0.000 [0.000,0.000]"
3,mean_pre_word_pause,"0.157 [0.070,0.688]","0.000 [0.000,0.000]","nan [nan,nan]","nan [nan,nan]","-0.000 [-0.000,0.000]","-0.170 [-0.178,-0.162]"
4,mean_turn_length_minutes,"0.070 [0.020,0.164]","0.070 [0.020,0.164]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","0.000 [0.000,0.000]"
5,mean_turn_length_words,"10.085 [2.885,21.516]","8.986 [2.656,18.198]","0.997 [0.996,0.997]","0.996 [0.994,0.997]","0.925 [0.913,0.935]","-1.155 [-1.219,-1.090]"
6,num_interrupts,"0.000 [0.000,28.000]","0.000 [0.000,28.000]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","0.000 [0.000,0.000]"
7,num_one_word_turns,"9.000 [1.000,28.000]","11.000 [1.000,31.000]","0.952 [0.937,0.965]","0.946 [0.927,0.962]","0.918 [0.895,0.938]","1.270 [1.080,1.469]"
8,num_turns,"106.000 [36.000,313.000]","106.000 [36.000,313.000]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","0.000 [0.000,0.000]"
9,phrase_repeat_percentage,"0.000 [0.000,0.000]","0.000 [0.000,0.000]","nan [nan,nan]","nan [nan,nan]","nan [nan,nan]","0.000 [0.000,0.000]"


In [3]:
L_SUMMARY_FEATURES = [
    "syllables_per_min",
    "sentiment_pos",
    "sentiment_neg",
    "sentiment_neu",
    "sentiment_overall",
    'sentiment_vader_pos', 'sentiment_vader_neg', 'sentiment_vader_neu',
       'sentiment_vader_overall',
    "mattr_5",
    "mattr_10",
    "mattr_25",
    "mattr_50",
    "mattr_100",
    "first_person_percentage",
    "first_person_sentiment_positive",
    "first_person_sentiment_negative",
    "first_person_sentiment_overall",
       'first_person_sentiment_positive_vader',
       'first_person_sentiment_negative_vader',
       'first_person_sentiment_overall_vader',
    "word_coherence_mean",
    "word_coherence_var",
    "word_coherence_5_mean",
    "word_coherence_5_var",
    "word_coherence_10_mean",
    "word_coherence_10_var",
    *[f"word_coherence_variability_{k}_mean" for k in range(2, 11)],
    *[f"word_coherence_variability_{k}_var" for k in range(2, 11)],
    "first_order_sentence_tangeniality_mean",
    "first_order_sentence_tangeniality_var",
    "second_order_sentence_tangeniality_mean",
    "second_order_sentence_tangeniality_var",
    "turn_to_turn_tangeniality_mean",
    "turn_to_turn_tangeniality_var",
    "turn_to_turn_tangeniality_slope",
    "semantic_perplexity_mean",
    "semantic_perplexity_var",
    "semantic_perplexity_5_mean",
    "semantic_perplexity_5_var",
    "semantic_perplexity_11_mean",
    "semantic_perplexity_11_var",
    "semantic_perplexity_15_mean",
    "semantic_perplexity_15_var",
]


stats_df = check_statistic(
    features=L_SUMMARY_FEATURES,
    dataset_eng_dir="tmp/result_phonoma_gemma_eng_26-03-2026",#"/Users/pelmeshek1706/Desktop/projects/airest_notebooks/result_oppenwillis_eng_gemma_new_json_split-1-3-26",#result_oppenwillis_eng_norm_gemma_updated_sentiment_FIXED_21-2-26",
    dataset_ukr_dir="tmp/result_phonoma_gemma_ukr_26-03-2026",#"/Users/pelmeshek1706/Desktop/projects/airest_notebooks/result_oppenwillis_ukr_gemma_new_json_split-1-3-26",#result_oppenwillis_ukr_norm_gemma_updated_sentiment_FIXED_21-2-26",
    file_kind="summary",
    n_boot=2000,
    seed=1706,
)
stats_df.drop(columns=["note", 'n'], inplace=True)
stats_df

,feature,"EN_median[min,max]","UA_median[min,max]",r [95% CI],rho [95% CI],"ICC(2,1) [95% CI]",mean shift Δμ (UA-EN) [95% CI]
0,first_person_percentage,"8.767 [3.852,12.865]","6.875 [3.358,10.672]","0.870 [0.839,0.898]","0.865 [0.830,0.896]","0.465 [0.414,0.515]","-1.870 [-1.963,-1.779]"
1,first_person_sentiment_negative,"2.356 [0.942,4.780]","1.852 [0.634,4.230]","0.899 [0.875,0.921]","0.878 [0.840,0.910]","0.700 [0.651,0.743]","-0.506 [-0.544,-0.465]"
2,first_person_sentiment_negative_vader,"0.388 [0.037,1.304]","0.335 [0.005,1.562]","0.807 [0.762,0.846]","0.815 [0.762,0.857]","0.788 [0.740,0.830]","-0.045 [-0.061,-0.029]"
3,first_person_sentiment_overall,"23.555 [5.383,39.756]","22.218 [9.377,39.764]","0.935 [0.917,0.950]","0.930 [0.910,0.946]","0.899 [0.876,0.921]","-1.441 [-1.660,-1.210]"
4,first_person_sentiment_overall_vader,"17.259 [8.679,33.070]","17.994 [7.695,32.707]","0.855 [0.815,0.887]","0.839 [0.794,0.876]","0.830 [0.790,0.865]","0.967 [0.713,1.229]"
5,first_person_sentiment_positive,"25.725 [13.849,40.996]","26.327 [15.431,41.739]","0.943 [0.926,0.956]","0.933 [0.913,0.950]","0.937 [0.920,0.952]","0.202 [-0.005,0.404]"
6,first_person_sentiment_positive_vader,"17.535 [8.449,32.395]","18.227 [7.622,32.918]","0.860 [0.822,0.892]","0.843 [0.800,0.881]","0.832 [0.793,0.866]","1.039 [0.794,1.295]"
7,mattr_10,"0.919 [0.843,0.951]","0.935 [0.846,0.975]","0.848 [0.806,0.883]","0.829 [0.780,0.870]","0.573 [0.504,0.640]","0.015 [0.014,0.016]"
8,mattr_100,"0.588 [0.458,0.672]","0.669 [0.574,0.775]","0.882 [0.852,0.910]","0.888 [0.860,0.914]","0.224 [0.192,0.257]","0.080 [0.078,0.081]"
9,mattr_25,"0.800 [0.687,0.863]","0.844 [0.761,0.897]","0.837 [0.792,0.877]","0.839 [0.791,0.881]","0.345 [0.301,0.390]","0.041 [0.039,0.043]"


In [4]:
L_TURNS = [
    "syllables_per_min",
    "sentiment_pos",
    "sentiment_neg",
    "sentiment_neu",
    "sentiment_overall",
    "mattr_5",
    "mattr_10",
    "mattr_25",
    "mattr_50",
    "mattr_100",
    "first_person_percentage",
    "first_person_sentiment_positive",
    "first_person_sentiment_negative",
    # discourse (per-turn)
    "first_order_sentence_tangeniality",
    "second_order_sentence_tangeniality",
    "turn_to_turn_tangeniality",
    "semantic_perplexity",
    "semantic_perplexity_5",
    "semantic_perplexity_11",
    "semantic_perplexity_15",
]

# 2) Turns files (turns_<pid>.csv), aggregated per participant by median:
stats_turns = check_statistic(
    features=L_TURNS,  # your per-turn feature list
    dataset_eng_dir="tmp/result_phonoma_gemma_eng_26-03-2026",#"/Users/pelmeshek1706/Desktop/projects/airest_notebooks/result_oppenwillis_eng_gemma_new_json_split-1-3-26",#result_oppenwillis_eng_norm_gemma_updated_sentiment_FIXED_21-2-26",
    dataset_ukr_dir="tmp/result_phonoma_gemma_ukr_26-03-2026",#"/Users/pelmeshek1706/Desktop/projects/airest_notebooks/result_oppenwillis_ukr_gemma_new_json_split-1-3-26",#result_oppenwillis_ukr_norm_gemma_updated_sentiment_FIXED_21-2-26",
    file_kind="turns",
    turns_agg="mean",
    # participant_only=True,
)

stats_turns.drop(columns=["note", 'n'], inplace=True)
stats_turns

,feature,"EN_median[min,max]","UA_median[min,max]",r [95% CI],rho [95% CI],"ICC(2,1) [95% CI]",mean shift Δμ (UA-EN) [95% CI]
0,first_person_percentage,"8.196 [3.344,12.830]","6.723 [2.191,10.554]","0.907 [0.884,0.927]","0.895 [0.862,0.923]","0.623 [0.571,0.670]","-1.567 [-1.655,-1.481]"
1,first_person_sentiment_negative,"2.356 [0.942,4.780]","1.852 [0.634,4.230]","0.899 [0.875,0.921]","0.878 [0.840,0.910]","0.700 [0.651,0.743]","-0.506 [-0.544,-0.465]"
2,first_person_sentiment_positive,"25.725 [13.849,40.996]","26.327 [15.431,41.739]","0.943 [0.926,0.956]","0.933 [0.913,0.950]","0.937 [0.920,0.952]","0.202 [-0.005,0.404]"
3,mattr_10,"0.959 [0.917,0.993]","0.971 [0.947,0.997]","0.816 [0.768,0.854]","0.789 [0.733,0.838]","0.504 [0.447,0.555]","0.012 [0.011,0.013]"
4,mattr_100,"0.929 [0.836,0.992]","0.954 [0.886,0.997]","0.937 [0.917,0.951]","0.923 [0.898,0.944]","0.577 [0.529,0.621]","0.024 [0.023,0.025]"
5,mattr_25,"0.933 [0.867,0.992]","0.957 [0.905,0.997]","0.916 [0.893,0.935]","0.904 [0.875,0.929]","0.547 [0.497,0.592]","0.022 [0.020,0.023]"
6,mattr_5,"0.983 [0.964,0.996]","0.988 [0.963,1.000]","0.623 [0.543,0.697]","0.587 [0.495,0.675]","0.490 [0.409,0.569]","0.004 [0.004,0.005]"
7,mattr_50,"0.929 [0.840,0.992]","0.955 [0.886,0.997]","0.933 [0.913,0.949]","0.920 [0.895,0.941]","0.573 [0.524,0.617]","0.024 [0.022,0.025]"
8,semantic_perplexity,"255.349 [40.046,33889.189]","370.450 [120.476,34837.182]","0.023 [-0.044,0.154]","0.201 [0.078,0.323]","0.010 [-0.028,0.086]","-102.457 [-471.505,259.694]"
9,semantic_perplexity_11,"990.702 [279.345,38075.138]","5797.996 [1850.641,41379.571]","0.023 [-0.086,0.155]","0.119 [-0.002,0.241]","0.007 [-0.016,0.037]","4567.930 [4136.221,4991.527]"
